# RGB Renk Segmentasyonu — Omar

Görüntü BGR okunur, RGB uzayına dönüştürülür ve `cv2.inRange` ile sabit aralıktaki pikseller seçilir.
Aralık **yalnızca G1/G2'de** seçilip dondurulmuştur; K1–K4 sonucu görüldükten sonra değiştirilmez.
- Maske alanı (%) = 100 × seçilen piksel / toplam piksel
- Baş örtüşmesi (%) = baş ROI içindeki seçili piksel oranı → dört sınıflı karar

In [ ]:
from pathlib import Path
import sys
KOK = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "kaynak" / "analiz.py").is_file())
sys.path.insert(0, str(KOK))

import cv2, numpy as np, pandas as pd
import matplotlib.pyplot as plt
from kaynak.veri import ayarlari_oku, veri_setini_oku, goruntu_oku, maske_oku
from kaynak.analiz import analiz_et
from kaynak.gorsel import bgr2rgb, kutulari_ciz, maske_bindir
ayarlar = ayarlari_oku()
kayitlar = veri_setini_oku()
pd.set_option("display.max_colwidth", 80)
print("Görüntüler:", ", ".join(f"{k['kimlik']} ({k['bolum']})" for k in kayitlar))

## 1. Dondurulmuş ayarlar ve karar kuralı

In [ ]:
print("RGB aralığı:", ayarlar["renk_araliklari"]["RGB"])
print("Karar kuralı:", ayarlar["karar_kurali"]["kural"])

## 2. Deney öncesi tahmin

In [ ]:
tahminler = pd.read_csv(KOK / "belgeler/tahminler.csv")
benim = tahminler[(tahminler.uye == "Omar") & tahminler.deney_kimligi.str.startswith("")]
if benim.empty:
    print("UYARI: Bu deney için kayıtlı tahmin yok. Hoca: 'Tahmini olmayan deney puanlanmaz.'")
    print("Deneyi çalıştırmadan ÖNCE belgeler/tahminler.csv dosyasına iki cümle ekleyin.")
for _, t in benim.iterrows():
    print(f"[{t.kayit_tarihi}] {t.deney_kimligi}\n 1) {t.tahmin_1}\n 2) {t.tahmin_2}")

## 3. Geliştirme (G1, G2) — aralık seçimi yalnızca burada

In [ ]:
def tablo(bolum):
    satirlar = []
    for k in veri_setini_oku(bolum):
        s = analiz_et(goruntu_oku(k["dosya"]), "RGB", ayarlar, k["bas_roi"], k["baret_kutulari"])
        satirlar.append({"Görüntü": k["kimlik"], "Koşul": k["kosul"], "Maske alanı (%)": round(s["alan_yuzde"], 2),
                         "Baş örtüşmesi (%)": None if s["ortusme_yuzde"] is None else round(s["ortusme_yuzde"], 2),
                         "Kuralın sınıfı": s["sinif"], "Doğru sınıf": k["dogru_sinif"],
                         "Doğru mu": s["sinif"] == k["dogru_sinif"]})
    return pd.DataFrame(satirlar)
tablo("gelistirme")

## 4. Kontrol (K1–K4) — 4 maske ve dört sınıflı karar

In [ ]:
kontrol = tablo("kontrol"); kontrol

In [ ]:
print(f"Sınıf doğruluğu: {kontrol['Doğru mu'].sum()}/{len(kontrol)}")

In [ ]:
fig, eksen = plt.subplots(2, 4, figsize=(20, 8))
for i, k in enumerate(veri_setini_oku("kontrol")):
    g = goruntu_oku(k["dosya"]); s = analiz_et(g, "RGB", ayarlar, k["bas_roi"])
    eksen[0, i].imshow(bgr2rgb(g)); eksen[0, i].set_title(k["kimlik"])
    eksen[1, i].imshow(s["maske"], cmap="gray"); eksen[1, i].set_title(f"RGB maske %{s['alan_yuzde']:.2f} → {s['sinif']}")
    for e in eksen[:, i]: e.axis("off")
plt.tight_layout(); plt.show()

## Hata yorumu, sınırlılık ve sonraki kontrol
*(Üye kendi gözlemini yazar: gölge, düşük doygunluk, aynı renkli nesne; renk maskesi baret nesnesini kanıtlamaz.)*